# Trees and TabPFN

Compare scores, data size, and total runtime. Is one method always best?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://github.com/PriorLabs/TabPFN

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors" "xgboost==2.1.4" "lightgbm==4.6.0" "tabpfn==2.2.1" "prophet==1.5.0" "chronos-forecasting==2.3.2"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import torch
torch.set_num_threads(1)
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import (
    load_breast_cancer, load_iris, load_wine, make_classification
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('🖥️ Environment ready')

In [ ]:
import xgboost as xgb

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

xgb_model = xgb.XGBClassifier(n_jobs=1, 
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    eval_metric='logloss',
)
xgb_model.fit(X_tr, y_tr)
preds = xgb_model.predict(X_te)
print(f'🌲 XGBoost accuracy: {accuracy_score(y_te, preds):.4f}')

In [ ]:
import lightgbm as lgb

train_set = lgb.Dataset(X_tr, label=y_tr)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'num_leaves': 31,         # leaf-wise control
    'learning_rate': 0.05,
    'feature_fraction': 0.8,  # column sampling
    'verbose': -1,
    'num_threads': 1,
}
lgb_model = lgb.train(params, train_set, num_boost_round=100)
preds_proba = lgb_model.predict(X_te)
preds = (preds_proba > 0.5).astype(int)
print(f'🌲 LightGBM accuracy: {accuracy_score(y_te, preds):.4f}')

In [ ]:
from tabpfn import TabPFNClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

# That's it — no hyperparameters to tune
tab_clf = TabPFNClassifier(device='cuda' if torch.cuda.is_available() else 'cpu', n_estimators=1, n_jobs=1, random_state=SEED)
tab_clf.fit(X_tr, y_tr)
preds = tab_clf.predict(X_te)
probs = tab_clf.predict_proba(X_te)

print(f'🧠 TabPFN accuracy: {accuracy_score(y_te, preds):.4f}')
print(f'   predict_proba shape: {probs.shape}')

## Demo code

Compare scores, data size, and total runtime. Is one method always best?

In [ ]:
model = TabPFNClassifier(device="cuda" if torch.cuda.is_available() else "cpu", n_estimators=1, random_state=42)
model.fit(X_tr, y_tr)
prediction = model.predict(X_te)
print("Held-out accuracy:", accuracy_score(y_te, prediction))

In [ ]:
print('━' * 60)
print('XGBoost (8 hyperparameters):')
print('━' * 60)
xgb_clf = xgb.XGBClassifier(n_jobs=1, 
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='logloss', verbosity=0,
)
t0 = time.time(); xgb_clf.fit(X_tr, y_tr); xgb_time = time.time() - t0
xgb_acc = accuracy_score(y_te, xgb_clf.predict(X_te))
print(f'  Accuracy: {xgb_acc:.4f}    Train time: {xgb_time:.2f}s')

print('\n' + '━' * 60)
print('TabPFN (0 hyperparameters):')
print('━' * 60)
tab_clf = TabPFNClassifier(device='cuda' if torch.cuda.is_available() else 'cpu', n_estimators=1, n_jobs=1, random_state=SEED)
t0 = time.time(); tab_clf.fit(X_tr, y_tr); fit_time = time.time() - t0
t0 = time.time(); tab_preds = tab_clf.predict(X_te); pred_time = time.time() - t0
tab_acc = accuracy_score(y_te, tab_preds)
print(f'  Accuracy: {tab_acc:.4f}    fit: {fit_time:.2f}s   predict: {pred_time:.2f}s')

print('Compare the measured scores and total fit + predict time on this fixed split.')

In [ ]:
X_full, y_full = make_classification(
    n_samples=3000, n_features=20, n_informative=15,
    n_redundant=2, n_classes=2, random_state=SEED,
)
X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_full, y_full, test_size=200, random_state=SEED, stratify=y_full,
)

sizes = [50, 100, 200, 500]
# A bounded classroom benchmark; extend only after timing the available hardware.
xgb_curve, lgb_curve, tab_curve = [], [], []

for n in sizes:
    X_n, y_n = X_tr_full[:n], y_tr_full[:n]
    m = xgb.XGBClassifier(n_jobs=1, n_estimators=100, max_depth=6, learning_rate=0.1, eval_metric='logloss', verbosity=0)
    m.fit(X_n, y_n)
    xgb_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    m = lgb.LGBMClassifier(n_jobs=1, n_estimators=100, num_leaves=31, learning_rate=0.05, verbose=-1)
    m.fit(X_n, y_n)
    lgb_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    m = TabPFNClassifier(device='cuda' if torch.cuda.is_available() else 'cpu', n_estimators=1, n_jobs=1, random_state=SEED)
    m.fit(X_n, y_n)
    tab_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    print(f'n={n:>5}: XGB={xgb_curve[-1]:.3f}  LGB={lgb_curve[-1]:.3f}  TabPFN={tab_curve[-1]:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes, xgb_curve, '-o', label='XGBoost (fixed settings)', color='#26C281', linewidth=2, markersize=8)
ax.plot(sizes, lgb_curve, '-s', label='LightGBM (fixed settings)', color='#7B5BE0', linewidth=2, markersize=8)
ax.plot(sizes, tab_curve, '-^', label='TabPFN v2 (1 estimator)', color='#26B7DB', linewidth=2, markersize=8)
# TabPFN v2 is pinned. This benchmark does not estimate the model's maximum capacity.
ax.set_xscale('log')
ax.set_xlabel('Training set size'); ax.set_ylabel('Accuracy')
ax.set_title('Performance vs Dataset Size: TabPFN vs Tree Models', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('This small synthetic benchmark is one task, not a general model ranking.')

In [ ]:
datasets = {
    'Breast Cancer': load_breast_cancer(return_X_y=True),
    'Iris':          load_iris(return_X_y=True),
    'Wine':          load_wine(return_X_y=True),
}

xgb_accs, lgb_accs = [], []
for name, (X, y) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    xm = xgb.XGBClassifier(n_jobs=1, 
        n_estimators=100, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', verbosity=0,
    )
    xm.fit(X_tr, y_tr)
    xgb_accs.append(accuracy_score(y_te, xm.predict(X_te)))

    lm = lgb.LGBMClassifier(n_jobs=1, 
        n_estimators=100, num_leaves=31, learning_rate=0.05, verbose=-1,
    )
    lm.fit(X_tr, y_tr)
    lgb_accs.append(accuracy_score(y_te, lm.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(datasets)); w = 0.35
ax.bar(x - w/2, xgb_accs, w, label='XGBoost', color='#26C281', edgecolor='black')
ax.bar(x + w/2, lgb_accs, w, label='LightGBM', color='#3498DB', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(list(datasets.keys()))
ax.set_ylabel('Accuracy'); ax.set_ylim(0.85, 1.02)
ax.set_title('XGBoost vs LightGBM — Real Benchmark Results', fontweight='bold')
for i, (xa, la) in enumerate(zip(xgb_accs, lgb_accs)):
    ax.text(i - w/2, xa + 0.005, f'{xa:.3f}', ha='center', fontweight='bold')
    ax.text(i + w/2, la + 0.005, f'{la:.3f}', ha='center', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

print('Compare the measured held-out scores on each task. This is not a general model ranking.')

## Interpretation

TabPFN v2 uses one estimator; the bounded CPU benchmark is not a model ranking.

Compare the output with the evidence in the input. A generated answer can be wrong.